# Notebook 06: Fine-Tuned Model Evaluation

Compare fine-tuned bi-encoder variants against the OpenAI baseline on both full-corpus retrieval and the hard-negative ablation.

**Prerequisite:** Run `01_index_apis.ipynb` and `05_train.ipynb` first.

In [ ]:
import os, sys, json
import numpy as np
from pathlib import Path
from dotenv import load_dotenv

REPO_ROOT = next(p for p in [Path().resolve()] + list(Path().resolve().parents) if (p / '.git').exists())
PROJECT_DIR = REPO_ROOT / 'project'
sys.path.insert(0, str(PROJECT_DIR))

load_dotenv(REPO_ROOT / '.env')

TOOLBENCH_DIR = Path(os.environ.get('TOOLBENCH_DIR', str(REPO_ROOT / 'toolbench_data')))

from data.load_toolbench import load_api_corpus, load_eval_examples
from data.negative_mining import build_api_lookup, build_random_negatives, build_category_sibling_negatives, build_dfsdt_negatives
from models.embeddings import format_api_string
from retrieval.retriever import build_faiss_index, retrieve_top_k
from evaluation.metrics import recall_at_k, mean_reciprocal_rank, evaluate_batch

In [ ]:
corpus = load_api_corpus(TOOLBENCH_DIR / 'toolenv' / 'tools')
lookup = build_api_lookup(corpus)
evals = load_eval_examples(TOOLBENCH_DIR / 'toolllama_G123_dfs_eval.json')

with open(PROJECT_DIR / 'api_names.json') as f:
    api_names = json.load(f)
name_to_idx = {name: i for i, name in enumerate(api_names)}

api_strings = [format_api_string(a) for a in corpus]
queries = [e['user_query'] for e in evals]

print(f'Corpus: {len(corpus)} APIs | Eval: {len(evals)} examples')

## Encode Corpus with Fine-Tuned Models

In [ ]:
from sentence_transformers import SentenceTransformer

models = {}
embeddings = {}

for label, path in [('v2_random', 'checkpoints/v2_random'), ('v3_hard', 'checkpoints/v3_hard')]:
    model_path = PROJECT_DIR / path
    if not model_path.exists():
        print(f'Skipping {label}: checkpoint not found at {model_path}')
        continue
    print(f'Encoding with {label}...')
    model = SentenceTransformer(str(model_path))
    models[label] = model
    embeddings[label] = {
        'corpus': model.encode(api_strings, show_progress_bar=True, batch_size=256),
        'queries': model.encode(queries, batch_size=256),
    }

print(f'Loaded {len(models)} fine-tuned models')

## Full-Corpus Evaluation

In [ ]:
with open(PROJECT_DIR / 'results_baseline.json') as f:
    baseline = json.load(f)['results']

full_corpus_results = {'baseline (text-embedding-3-small)': baseline}

for label, embs in embeddings.items():
    index = build_faiss_index(embs['corpus'])
    all_retrieved, all_gt = [], []
    for i, ex in enumerate(evals):
        top_k = retrieve_top_k(embs['queries'][i], index, k=10)
        all_retrieved.append(top_k)
        all_gt.append([name_to_idx[n] for n in ex['ground_truth_apis'] if n in name_to_idx])
    full_corpus_results[label] = evaluate_batch(all_retrieved, all_gt, ks=[1, 5, 10])

print(f"{'Model':<40} {'R@1':>8} {'R@5':>8} {'R@10':>8} {'MRR':>8}")
print('-' * 72)
for label, r in full_corpus_results.items():
    print(f"{label:<40} {r['recall@1']:>8.4f} {r['recall@5']:>8.4f} {r['recall@10']:>8.4f} {r['mrr']:>8.4f}")

## Hard-Negative Ablation

Same restricted 100-candidate pool evaluation as notebook 03, now comparing baseline vs fine-tuned models.

In [ ]:
with open(TOOLBENCH_DIR / 'toolllama_G123_dfs_eval.json') as f:
    raw_evals = json.load(f)

def eval_restricted_pool(query_embs, corpus_embs, neg_type, n_neg=99, ks=[1, 5, 10]):
    recall_scores = {k: [] for k in ks}
    mrr_scores = []
    for i, ex in enumerate(evals):
        raw_ex = raw_evals[ex['raw_idx']]
        gt_names = ex['ground_truth_apis']
        gt_indices = [name_to_idx[n] for n in gt_names if n in name_to_idx]
        if not gt_indices:
            continue
        if neg_type == 'random':
            negs = build_random_negatives(corpus, gt_names, n=n_neg)
        elif neg_type == 'sibling':
            negs = build_category_sibling_negatives(corpus, gt_names, lookup, n=n_neg)
        else:
            negs = build_dfsdt_negatives(raw_ex, corpus, gt_names, lookup, n=n_neg)
        cand_indices = [name_to_idx[n] for n in gt_names + [a['action_name'] for a in negs] if n in name_to_idx]
        if len(cand_indices) < 2:
            continue
        local_index = build_faiss_index(corpus_embs[cand_indices].astype(np.float32))
        g2l = {g: j for j, g in enumerate(cand_indices)}
        local_gt = [g2l[g] for g in gt_indices if g in g2l]
        if not local_gt:
            continue
        top_k = retrieve_top_k(query_embs[i], local_index, k=min(10, len(cand_indices)))
        for k in ks:
            recall_scores[k].append(recall_at_k(top_k, local_gt, k))
        mrr_scores.append(mean_reciprocal_rank(top_k, local_gt))
    return {**{f'recall@{k}': float(np.mean(recall_scores[k])) for k in ks}, 'mrr': float(np.mean(mrr_scores))}

In [ ]:
with open(PROJECT_DIR / 'results_hard_negatives.json') as f:
    baseline_hn = json.load(f)

hn_results = {'baseline': baseline_hn}
for label, embs in embeddings.items():
    hn_results[label] = {}
    for neg_type in ['random', 'sibling', 'dfsdt']:
        print(f'{label} / {neg_type}...')
        hn_results[label][neg_type] = eval_restricted_pool(embs['queries'], embs['corpus'], neg_type)

print(f"\n{'Model / Condition':<45} {'R@1':>8} {'R@5':>8} {'R@10':>8} {'MRR':>8}")
print('-' * 77)
for model_label, conditions in hn_results.items():
    for cond, r in conditions.items():
        print(f"{model_label} / {cond:<20} {r['recall@1']:>8.4f} {r['recall@5']:>8.4f} {r['recall@10']:>8.4f} {r['mrr']:>8.4f}")

## Save Results

In [ ]:
results = {
    'full_corpus': {k: v for k, v in full_corpus_results.items()},
    'hard_negative_ablation': hn_results,
}
with open(PROJECT_DIR / 'results_finetuned.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Saved results_finetuned.json')